# STEP 32 — IEEE Performance Diagnostics Suite

This notebook performs a transparent diagnostic analysis of the subject-level BrainFMOps-Analyze results.

It does **not** change the model and does **not** alter ROC-AUC. It evaluates how the operating threshold affects threshold-dependent metrics and generates error-analysis artifacts for the manuscript.

## Outputs

- Threshold–performance curve
- Sensitivity–specificity curve
- Precision–recall curve
- Calibration curve
- Error probability distribution
- Confidence-band analysis
- Best-threshold table
- False-positive and false-negative case lists
- Summary report for Section 5 and Discussion

## Scientific safeguard

A threshold optimized on this labeled cohort must be reported as a **post hoc operating-point analysis**, not as independent test-set optimization. Independent threshold selection requires a separate validation set.

## Compatibility

Version 2 includes fallbacks for older/newer scikit-learn, NumPy 2.x, and Matplotlib boxplot argument changes. No path editing is required.


In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    brier_score_loss,
)

try:
    from sklearn.calibration import calibration_curve
except ImportError:
    calibration_curve = None

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 600
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 9

print("Environment ready.")


## 1. Automatically locate the labeled subject-level evaluation file


In [ ]:
ROOT_CANDIDATES = [
    Path.cwd().resolve(),
    Path.cwd().resolve(),
    Path.cwd(),
]

ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), Path.cwd())

candidate_files = [
    ROOT / "31B_GroundTruth_Integration" / "evaluation_summary_with_labels.csv",
    ROOT / "evaluation_summary_with_labels.csv",
]

CSV = next((p for p in candidate_files if p.is_file()), None)

if CSV is None:
    matches = list(ROOT.rglob("evaluation_summary_with_labels.csv"))
    CSV = matches[0] if matches else None

if CSV is None:
    raise FileNotFoundError(
        "evaluation_summary_with_labels.csv was not found inside the BrainFMOps project."
    )

OUT = ROOT / "32_IEEE_Performance_Diagnostics"
OUT.mkdir(parents=True, exist_ok=True)

LOCKED_THRESHOLD = 0.32
THRESHOLD_GRID = np.round(np.arange(0.00, 1.001, 0.01), 2)
DPI = 600

print("Project root:", ROOT)
print("Input file:", CSV)
print("Output directory:", OUT)


## 2. Load and validate labeled data


In [ ]:
df = pd.read_csv(CSV, encoding="utf-8-sig")

required = {"ground_truth", "probability_positive"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

if "case_id" not in df.columns:
    if "subject_id" in df.columns:
        df["case_id"] = df["subject_id"]
    else:
        df["case_id"] = np.arange(len(df)).astype(str)

df = df[df["ground_truth"].isin(["CN", "AD"])].copy()
df["y_true"] = (df["ground_truth"] == "AD").astype(int)
df["y_prob"] = pd.to_numeric(df["probability_positive"], errors="coerce")
df = df[df["y_prob"].between(0, 1, inclusive="both")].copy()

if df["y_true"].nunique() < 2:
    raise ValueError("Both CN and AD ground-truth classes are required.")

print("Evaluated labeled subjects:", len(df))
print(df["ground_truth"].value_counts())
print("ROC-AUC:", roc_auc_score(df["y_true"], df["y_prob"]))
print("PR-AUC:", average_precision_score(df["y_true"], df["y_prob"]))


## 3. Threshold sweep


In [ ]:
def metric_row(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    balanced_accuracy = np.nanmean([sensitivity, specificity])
    youden_j = sensitivity + specificity - 1

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy,
        "youden_j": youden_j,
        "npv": npv,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

threshold_df = pd.DataFrame([
    metric_row(df["y_true"].to_numpy(), df["y_prob"].to_numpy(), t)
    for t in THRESHOLD_GRID
])

threshold_df.to_csv(
    OUT / "threshold_performance_all.csv",
    index=False,
    encoding="utf-8-sig"
)

display(threshold_df.head())
display(threshold_df.tail())


## 4. Identify operating points


In [ ]:
def best_row(metric):
    idx = threshold_df[metric].idxmax()
    return threshold_df.loc[idx]

best_youden = best_row("youden_j")
best_f1 = best_row("f1_score")
best_balanced = best_row("balanced_accuracy")
locked = threshold_df.loc[
    np.isclose(threshold_df["threshold"], LOCKED_THRESHOLD)
].iloc[0]

operating_points = pd.DataFrame([
    {"criterion": "Locked threshold", **locked.to_dict()},
    {"criterion": "Maximum Youden J", **best_youden.to_dict()},
    {"criterion": "Maximum F1-score", **best_f1.to_dict()},
    {"criterion": "Maximum balanced accuracy", **best_balanced.to_dict()},
])

operating_points.to_csv(
    OUT / "Table_V_Operating_Point_Analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

display(operating_points[[
    "criterion", "threshold", "accuracy", "precision",
    "sensitivity", "specificity", "f1_score",
    "balanced_accuracy", "youden_j", "tn", "fp", "fn", "tp"
]])


## 5. Fig. 20 — Threshold–performance curve

The locked threshold and the post hoc Youden operating point are shown separately.


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))

ax.plot(threshold_df["threshold"], threshold_df["accuracy"], label="Accuracy")
ax.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
ax.plot(threshold_df["threshold"], threshold_df["sensitivity"], label="Sensitivity")
ax.plot(threshold_df["threshold"], threshold_df["specificity"], label="Specificity")
ax.plot(threshold_df["threshold"], threshold_df["f1_score"], label="F1-score")
ax.plot(threshold_df["threshold"], threshold_df["balanced_accuracy"], label="Balanced accuracy")

ax.axvline(
    LOCKED_THRESHOLD,
    linestyle="--",
    linewidth=1.3,
    label=f"Locked threshold = {LOCKED_THRESHOLD:.2f}"
)
ax.axvline(
    best_youden["threshold"],
    linestyle=":",
    linewidth=1.5,
    label=f"Post hoc Youden threshold = {best_youden['threshold']:.2f}"
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Metric value")
ax.set_title("Subject-Level Performance Across Decision Thresholds")
ax.grid(alpha=0.2)
ax.legend(ncol=2)
fig.tight_layout()

fig.savefig(OUT / "Fig20_Threshold_Performance_Curve.png", dpi=DPI, bbox_inches="tight")
fig.savefig(OUT / "Fig20_Threshold_Performance_Curve.tiff", dpi=DPI, bbox_inches="tight")
plt.show()


## 6. Fig. 21 — Sensitivity and specificity trade-off


In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4.6))

ax.plot(threshold_df["threshold"], threshold_df["sensitivity"], label="Sensitivity")
ax.plot(threshold_df["threshold"], threshold_df["specificity"], label="Specificity")
ax.axvline(
    LOCKED_THRESHOLD,
    linestyle="--",
    linewidth=1.3,
    label=f"Locked threshold = {LOCKED_THRESHOLD:.2f}"
)
ax.axvline(
    best_youden["threshold"],
    linestyle=":",
    linewidth=1.5,
    label=f"Maximum Youden J = {best_youden['threshold']:.2f}"
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Rate")
ax.set_title("Sensitivity–Specificity Trade-off")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()

fig.savefig(OUT / "Fig21_Sensitivity_Specificity_Tradeoff.png", dpi=DPI, bbox_inches="tight")
fig.savefig(OUT / "Fig21_Sensitivity_Specificity_Tradeoff.tiff", dpi=DPI, bbox_inches="tight")
plt.show()


## 7. Fig. 22 — Precision–recall curve


In [ ]:
precision_values, recall_values, pr_thresholds = precision_recall_curve(
    df["y_true"], df["y_prob"]
)
pr_auc = average_precision_score(df["y_true"], df["y_prob"])
prevalence = df["y_true"].mean()

fig, ax = plt.subplots(figsize=(5.6, 4.7))
ax.plot(recall_values, precision_values, linewidth=1.8, label=f"PR-AUC = {pr_auc:.3f}")
ax.axhline(prevalence, linestyle="--", linewidth=1.0, label=f"Prevalence = {prevalence:.3f}")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Subject-Level Precision–Recall Curve")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()

fig.savefig(OUT / "Fig22_Precision_Recall_Curve.png", dpi=DPI, bbox_inches="tight")
fig.savefig(OUT / "Fig22_Precision_Recall_Curve.tiff", dpi=DPI, bbox_inches="tight")
plt.show()

pd.DataFrame({
    "precision": precision_values,
    "recall": recall_values,
    "threshold": np.append(pr_thresholds, np.nan)
}).to_csv(OUT / "Fig22_PR_curve_points.csv", index=False, encoding="utf-8-sig")


## 8. Fig. 23 — Calibration analysis


In [ ]:
def calibration_curve_fallback(y_true, y_prob, n_bins=10):
    table = pd.DataFrame({"y_true": np.asarray(y_true), "y_prob": np.asarray(y_prob)})
    try:
        table["bin"] = pd.qcut(
            table["y_prob"], q=n_bins, duplicates="drop"
        )
    except ValueError:
        table["bin"] = pd.cut(
            table["y_prob"], bins=n_bins, include_lowest=True, duplicates="drop"
        )

    grouped = table.groupby("bin", observed=False)
    summary = grouped.agg(
        mean_predicted_probability=("y_prob", "mean"),
        observed_positive_fraction=("y_true", "mean"),
        subjects=("y_true", "size"),
    ).dropna()

    return (
        summary["observed_positive_fraction"].to_numpy(),
        summary["mean_predicted_probability"].to_numpy(),
        summary.reset_index(drop=True),
    )

if calibration_curve is not None:
    prob_true, prob_pred = calibration_curve(
        df["y_true"], df["y_prob"], n_bins=10, strategy="quantile"
    )
    calibration_table = pd.DataFrame({
        "mean_predicted_probability": prob_pred,
        "observed_positive_fraction": prob_true,
    })
    calibration_method = "sklearn.calibration.calibration_curve"
else:
    prob_true, prob_pred, calibration_table = calibration_curve_fallback(
        df["y_true"], df["y_prob"], n_bins=10
    )
    calibration_method = "internal quantile-bin fallback"

brier = brier_score_loss(df["y_true"], df["y_prob"])
print("Calibration method:", calibration_method)

fig, ax = plt.subplots(figsize=(5.6, 4.7))
ax.plot(prob_pred, prob_true, marker="o", linewidth=1.5, label=f"Model (Brier = {brier:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.0, label="Perfect calibration")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed AD proportion")
ax.set_title("Subject-Level Calibration Plot")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()

fig.savefig(OUT / "Fig23_Calibration_Plot.png", dpi=DPI, bbox_inches="tight")
fig.savefig(OUT / "Fig23_Calibration_Plot.tiff", dpi=DPI, bbox_inches="tight")
plt.show()

calibration_table.to_csv(
    OUT / "Fig23_Calibration_points.csv",
    index=False,
    encoding="utf-8-sig"
)


## 9. Error analysis at the locked threshold


In [ ]:
df["y_pred_locked"] = (df["y_prob"] >= LOCKED_THRESHOLD).astype(int)

def error_type(row):
    if row["y_true"] == 0 and row["y_pred_locked"] == 0:
        return "TN"
    if row["y_true"] == 0 and row["y_pred_locked"] == 1:
        return "FP"
    if row["y_true"] == 1 and row["y_pred_locked"] == 0:
        return "FN"
    return "TP"

df["error_type"] = df.apply(error_type, axis=1)
df["distance_from_threshold"] = (df["y_prob"] - LOCKED_THRESHOLD).abs()

false_positive = df[df["error_type"] == "FP"].sort_values(
    "distance_from_threshold", ascending=False
)
false_negative = df[df["error_type"] == "FN"].sort_values(
    "distance_from_threshold", ascending=False
)

false_positive.to_csv(OUT / "false_positive_cases.csv", index=False, encoding="utf-8-sig")
false_negative.to_csv(OUT / "false_negative_cases.csv", index=False, encoding="utf-8-sig")
df.to_csv(OUT / "subject_level_diagnostic_records.csv", index=False, encoding="utf-8-sig")

print(df["error_type"].value_counts())
print("\nHighest-confidence false positives:")
display(false_positive[["case_id", "ground_truth", "y_prob", "distance_from_threshold"]].head(10))
print("\nHighest-confidence false negatives:")
display(false_negative[["case_id", "ground_truth", "y_prob", "distance_from_threshold"]].head(10))


## 10. Fig. 24 — Probability distribution by prediction outcome


In [ ]:
order = ["TN", "FP", "FN", "TP"]
data_groups = [
    df.loc[df["error_type"] == group, "y_prob"].to_numpy()
    for group in order
]

fig, ax = plt.subplots(figsize=(6.6, 4.7))
try:
    ax.boxplot(data_groups, tick_labels=order, showmeans=True)
except TypeError:
    ax.boxplot(data_groups, labels=order, showmeans=True)
ax.axhline(
    LOCKED_THRESHOLD,
    linestyle="--",
    linewidth=1.3,
    label=f"Locked threshold = {LOCKED_THRESHOLD:.2f}"
)
ax.set_ylim(0, 1)
ax.set_xlabel("Prediction outcome")
ax.set_ylabel("Predicted AD probability")
ax.set_title("Probability Distribution Across Prediction Outcomes")
ax.grid(axis="y", alpha=0.2)
ax.legend()
fig.tight_layout()

fig.savefig(OUT / "Fig24_Error_Probability_Distribution.png", dpi=DPI, bbox_inches="tight")
fig.savefig(OUT / "Fig24_Error_Probability_Distribution.tiff", dpi=DPI, bbox_inches="tight")
plt.show()


## 11. Confidence-band analysis


In [ ]:
# Confidence is defined relative to the locked operating threshold.
df["confidence_margin"] = (df["y_prob"] - LOCKED_THRESHOLD).abs()

df["confidence_band"] = pd.cut(
    df["confidence_margin"],
    bins=[-np.inf, 0.05, 0.15, np.inf],
    labels=["Low (≤0.05)", "Medium (0.05–0.15)", "High (>0.15)"]
)

confidence_summary = (
    df.groupby("confidence_band", observed=False)
    .agg(
        subjects=("case_id", "count"),
        correct=("error_type", lambda s: int(s.isin(["TN", "TP"]).sum())),
        incorrect=("error_type", lambda s: int(s.isin(["FP", "FN"]).sum())),
        mean_probability=("y_prob", "mean"),
    )
    .reset_index()
)

confidence_summary["accuracy"] = (
    confidence_summary["correct"] / confidence_summary["subjects"]
)

confidence_summary.to_csv(
    OUT / "Table_VI_Confidence_Band_Analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

display(confidence_summary)


## 12. Fig. 25 — Confidence-band accuracy


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.5))
x = np.arange(len(confidence_summary))
ax.bar(x, confidence_summary["accuracy"])
ax.set_xticks(x)
ax.set_xticklabels(confidence_summary["confidence_band"], rotation=15)
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_xlabel("Confidence band")
ax.set_title("Classification Accuracy by Confidence Margin")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()

fig.savefig(OUT / "Fig25_Confidence_Band_Accuracy.png", dpi=DPI, bbox_inches="tight")
fig.savefig(OUT / "Fig25_Confidence_Band_Accuracy.tiff", dpi=DPI, bbox_inches="tight")
plt.show()


## 13. Generate manuscript-ready diagnostic summary


In [ ]:
roc_auc = roc_auc_score(df["y_true"], df["y_prob"])
pr_auc = average_precision_score(df["y_true"], df["y_prob"])

summary = f"""BrainFMOps-Analyze — STEP 32 Performance Diagnostics
====================================================================
Labeled subjects: {len(df)}
CN subjects: {(df['y_true'] == 0).sum()}
AD subjects: {(df['y_true'] == 1).sum()}

Threshold-independent discrimination
------------------------------------
ROC-AUC: {roc_auc:.3f}
PR-AUC: {pr_auc:.3f}
Brier score: {brier:.3f}

Locked operating point
----------------------
Threshold: {locked['threshold']:.2f}
Accuracy: {locked['accuracy']:.3f}
Precision: {locked['precision']:.3f}
Sensitivity: {locked['sensitivity']:.3f}
Specificity: {locked['specificity']:.3f}
F1-score: {locked['f1_score']:.3f}
Balanced accuracy: {locked['balanced_accuracy']:.3f}
Youden J: {locked['youden_j']:.3f}

Post hoc maximum-Youden operating point
---------------------------------------
Threshold: {best_youden['threshold']:.2f}
Accuracy: {best_youden['accuracy']:.3f}
Precision: {best_youden['precision']:.3f}
Sensitivity: {best_youden['sensitivity']:.3f}
Specificity: {best_youden['specificity']:.3f}
F1-score: {best_youden['f1_score']:.3f}
Balanced accuracy: {best_youden['balanced_accuracy']:.3f}
Youden J: {best_youden['youden_j']:.3f}

Interpretation boundary
-----------------------
The optimized operating points are post hoc analyses on the labeled evaluation cohort.
They must not be described as independently validated thresholds. ROC-AUC remains
unchanged by threshold selection and should be used to characterize the model's
underlying ranking discrimination.
"""

print(summary)
(OUT / "STEP32_manuscript_diagnostic_summary.txt").write_text(
    summary, encoding="utf-8"
)


## 14. Output manifest


In [ ]:
generated_files = sorted(
    str(p.relative_to(OUT))
    for p in OUT.rglob("*")
    if p.is_file()
)

manifest = {
    "step": "32",
    "input_file": str(CSV),
    "output_directory": str(OUT),
    "labeled_subjects": int(len(df)),
    "locked_threshold": LOCKED_THRESHOLD,
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "brier_score": float(brier),
    "post_hoc_youden_threshold": float(best_youden["threshold"]),
    "scientific_boundary": (
        "Post hoc threshold analysis is descriptive and does not replace "
        "independent validation-set threshold selection."
    ),
    "generated_files": generated_files,
}

(OUT / "STEP32_output_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Completed.")
print("Output directory:", OUT)
for name in generated_files:
    print(" -", name)
